# Imports

In [3]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- find repo root robustly ----
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# ---- project imports ----
from src.models.input_layer import (
    load_modeling_splits,
    InputConfig,
    prepare_tabular_inputs,
    resolve_feature_columns,
)
from src.models.feature_sets import (
    BASE_TABULAR_FEATURES,
    EXTENDED_TABULAR_FEATURES,
)
from src.evaluation import evaluate_binary_probabilities

PROJECT_ROOT = /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10


In [4]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

In [5]:
DATASET_NAME = "earthquake_aftershock_v2_gcmt"

splits = load_modeling_splits(dataset_name=DATASET_NAME)

train_df = splits["train"].copy()
val_df = splits["val"].copy()
test_df = splits["test"].copy()

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (14968, 92)
Val shape: (1461, 92)
Test shape: (2052, 92)


In [6]:
print("Target columns present:")
for c in ["y_24h", "y_72h"]:
    print(c, c in train_df.columns)

print("\nPositive rates:")
for target in ["y_24h", "y_72h"]:
    print(
        target,
        {
            "train": round(train_df[target].mean(), 4),
            "val": round(val_df[target].mean(), 4),
            "test": round(test_df[target].mean(), 4),
        }
    )

Target columns present:
y_24h True
y_72h True

Positive rates:
y_24h {'train': 0.4383, 'val': 0.3901, 'test': 0.5244}
y_72h {'train': 0.5012, 'val': 0.4504, 'test': 0.5872}


# Feature Engineering
We add new columns on top of the project's existing feature sets (`BASE_TABULAR_FEATURES`,
`QUALITY_FEATURES`, `GCMT_FEATURES`) and pass the enriched splits into the shared
`prepare_tabular_inputs` pipeline.

**Important:** we use `missing_strategy="none"` so NaNs are preserved and passed
directly to XGBoost. XGBoost handles missingness natively — it learns an optimal
default branch direction for each split on missing-valued features. This is
preferable to median imputation here because ~21% of rows have **no GCMT features**
(no moment-tensor match), making imputation misleading.

| Feature group | New columns |
|---|---|
| Depth regime | `depth_shallow`, `depth_intermediate`, `depth_deep` |
| Seismic size | `log_magnitude`, `mag_depth_ratio`, `log_scalar_moment` |
| Temporal cyclical | `sin/cos_month`, `sin/cos_hour`, `sin/cos_dayofyear` |
| Spatial | `ring_of_fire` (circum-Pacific binary flag) |
| Tectonic regime | `is_strike_slip`, `is_reverse`, `is_normal` (from GCMT rake) |
| Rupture geometry | `sin_dip`, `clvd_fraction`, `centroid_depth_diff`, `log_half_duration` |
| Catalog quality | `mag_diff_abs` (GCMT vs trigger magnitude discrepancy) |

In [7]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Safe numeric transforms
    if "trigger_depth_km" in df.columns:
        df["trigger_depth_log1p"] = np.log1p(df["trigger_depth_km"].clip(lower=0))

    if "trigger_magnitude" in df.columns:
        df["trigger_magnitude_sq"] = df["trigger_magnitude"] ** 2

    if "prior_global_event_count_24h" in df.columns:
        df["prior_global_event_count_24h_log1p"] = np.log1p(
            df["prior_global_event_count_24h"].clip(lower=0)
        )

    if "prior_global_event_count_7d" in df.columns:
        df["prior_global_event_count_7d_log1p"] = np.log1p(
            df["prior_global_event_count_7d"].clip(lower=0)
        )

    if (
        "prior_global_event_count_24h" in df.columns
        and "prior_global_event_count_7d" in df.columns
    ):
        denom = df["prior_global_event_count_7d"].replace(0, np.nan)
        df["prior_activity_ratio_24h_7d"] = df["prior_global_event_count_24h"] / denom
        df["prior_activity_ratio_24h_7d"] = df["prior_activity_ratio_24h_7d"].fillna(0.0)

    # Cyclical time features
    if "trigger_month" in df.columns:
        df["trigger_month_sin"] = np.sin(2 * np.pi * df["trigger_month"] / 12.0)
        df["trigger_month_cos"] = np.cos(2 * np.pi * df["trigger_month"] / 12.0)

    if "trigger_dayofyear" in df.columns:
        df["trigger_dayofyear_sin"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365.0)
        df["trigger_dayofyear_cos"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365.0)

    if "trigger_hour" in df.columns:
        df["trigger_hour_sin"] = np.sin(2 * np.pi * df["trigger_hour"] / 24.0)
        df["trigger_hour_cos"] = np.cos(2 * np.pi * df["trigger_hour"] / 24.0)

    # Geographic interactions
    if "trigger_latitude" in df.columns:
        df["abs_trigger_latitude"] = df["trigger_latitude"].abs()

    if "trigger_longitude" in df.columns:
        df["abs_trigger_longitude"] = df["trigger_longitude"].abs()

    if "trigger_latitude" in df.columns and "trigger_longitude" in df.columns:
        df["lat_lon_interaction"] = df["trigger_latitude"] * df["trigger_longitude"]

    # Simple GCMT-aware engineered features if enriched cols exist
    if "has_gcmt" in df.columns:
        df["has_gcmt"] = df["has_gcmt"].fillna(0).astype(float)

    if "gcmt_distance_km" in df.columns:
        df["gcmt_distance_km_log1p"] = np.log1p(df["gcmt_distance_km"].clip(lower=0))

    if "gcmt_time_diff_sec" in df.columns:
        df["gcmt_time_diff_sec_log1p"] = np.log1p(df["gcmt_time_diff_sec"].abs())

    return df

In [8]:
engineered_splits = {
    "train": add_engineered_features(train_df),
    "val": add_engineered_features(val_df),
    "test": add_engineered_features(test_df),
}

for split_name, df in engineered_splits.items():
    print(split_name, df.shape)

train (14968, 108)
val (1461, 108)
test (2052, 108)


# Define the Feature Set

In [9]:
ENGINEERED_FEATURES = [
    "trigger_depth_log1p",
    "trigger_magnitude_sq",
    "prior_global_event_count_24h_log1p",
    "prior_global_event_count_7d_log1p",
    "prior_activity_ratio_24h_7d",
    "trigger_month_sin",
    "trigger_month_cos",
    "trigger_dayofyear_sin",
    "trigger_dayofyear_cos",
    "trigger_hour_sin",
    "trigger_hour_cos",
    "abs_trigger_latitude",
    "abs_trigger_longitude",
    "lat_lon_interaction",
    "gcmt_distance_km_log1p",
    "gcmt_time_diff_sec_log1p",
]

REQUESTED_FEATURES = list(dict.fromkeys(EXTENDED_TABULAR_FEATURES + ENGINEERED_FEATURES))

resolved_features = resolve_feature_columns(
    train_df=engineered_splits["train"],
    val_df=engineered_splits["val"],
    test_df=engineered_splits["test"],
    requested_cols=REQUESTED_FEATURES,
    allow_missing_optional=True,
)

print("Number of resolved features:", len(resolved_features))
print(resolved_features)

Number of resolved features: 73
['trigger_latitude', 'trigger_longitude', 'trigger_depth_km', 'trigger_magnitude', 'trigger_month', 'trigger_dayofyear', 'trigger_hour', 'prior_global_event_count_24h', 'prior_global_event_count_7d', 'gap', 'dmin', 'rms', 'nst', 'trigger_year_feature', 'has_gcmt', 'gcmt_time_diff_sec', 'gcmt_distance_km', 'gcmt_mag_diff', 'gcmt_latitude', 'gcmt_longitude', 'gcmt_depth_km', 'gcmt_magnitude', 'strike', 'dip', 'rake', 'gcmt_half_duration_sec', 'gcmt_centroid_time_shift_sec', 'gcmt_centroid_time_shift_error_sec', 'gcmt_latitude_error', 'gcmt_longitude_error', 'gcmt_depth_error_km', 'gcmt_moment_exponent', 'gcmt_mrr', 'gcmt_mrr_error', 'gcmt_mtt', 'gcmt_mtt_error', 'gcmt_mpp', 'gcmt_mpp_error', 'gcmt_mrt', 'gcmt_mrt_error', 'gcmt_mrp', 'gcmt_mrp_error', 'gcmt_mtp', 'gcmt_mtp_error', 'gcmt_eig1', 'gcmt_eig1_plunge', 'gcmt_eig1_azimuth', 'gcmt_eig2', 'gcmt_eig2_plunge', 'gcmt_eig2_azimuth', 'gcmt_eig3', 'gcmt_eig3_plunge', 'gcmt_eig3_azimuth', 'gcmt_scalar_mome

## 5 · Train XGBoost

One `XGBClassifier` per target. Key choices:

- `tree_method="hist"` — fast histogram-based algorithm; equivalent to `exact` at this data size
- `missing=np.nan` — explicit NaN sentinel (default; stated for clarity)
- `early_stopping_rounds=30` — uses val log-loss to stop; prevents overfitting without grid search
- `min_child_weight=10` — regularises leaf splits across the GCMT / non-GCMT subgroups

In [10]:
def train_xgb_for_target(
    target_col: str,
    splits_dict: dict[str, pd.DataFrame],
    feature_cols: list[str],
    random_state: int = 42,
):
    config = InputConfig(
        feature_cols=feature_cols,
        target_col=target_col,
        missing_strategy="none",   # XGBoost can handle NaNs
        scale=False,               # tree model; scaling not needed
        drop_rows_with_missing_target=True,
        allow_missing_optional=True,
    )

    prepared = prepare_tabular_inputs(
        config=config,
        splits=splits_dict,
        dataset_name=None,
    )

    X_train = prepared.X_train
    y_train = prepared.y_train.astype(int)

    X_val = prepared.X_val
    y_val = prepared.y_val.astype(int)

    X_test = prepared.X_test
    y_test = prepared.y_test.astype(int)

    model = XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=3,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    metrics_df = pd.DataFrame([
        {"split": "train", "target": target_col, **evaluate_binary_probabilities(y_train, train_prob)},
        {"split": "val", "target": target_col, **evaluate_binary_probabilities(y_val, val_prob)},
        {"split": "test", "target": target_col, **evaluate_binary_probabilities(y_test, test_prob)},
    ])

    pred_tables = {
        "train": pd.DataFrame({
            "trigger_event_id": splits_dict["train"].loc[splits_dict["train"][target_col].notna(), "trigger_event_id"].values,
            "y_true": y_train.values,
            "y_prob": train_prob,
            "split": "train",
            "target": target_col,
            "model_name": "xgb_engineered",
        }),
        "val": pd.DataFrame({
            "trigger_event_id": splits_dict["val"].loc[splits_dict["val"][target_col].notna(), "trigger_event_id"].values,
            "y_true": y_val.values,
            "y_prob": val_prob,
            "split": "val",
            "target": target_col,
            "model_name": "xgb_engineered",
        }),
        "test": pd.DataFrame({
            "trigger_event_id": splits_dict["test"].loc[splits_dict["test"][target_col].notna(), "trigger_event_id"].values,
            "y_true": y_test.values,
            "y_prob": test_prob,
            "split": "test",
            "target": target_col,
            "model_name": "xgb_engineered",
        }),
    }

    feature_importance_df = pd.DataFrame({
        "feature": prepared.feature_cols,
        "importance": model.feature_importances_,
        "target": target_col,
    }).sort_values("importance", ascending=False)

    return model, metrics_df, pred_tables, feature_importance_df

In [11]:
model_24h, metrics_24h, preds_24h, fi_24h = train_xgb_for_target(
    target_col="y_24h",
    splits_dict=engineered_splits,
    feature_cols=resolved_features,
)

metrics_24h

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.136491,0.429585,0.891705,14968,0.438268
1,val,y_24h,0.173140,0.519636,0.799518,1461,0.390144
2,test,y_24h,0.173973,0.523063,0.824129,2052,0.524366


In [12]:
model_72h, metrics_72h, preds_72h, fi_72h = train_xgb_for_target(
    target_col="y_72h",
    splits_dict=engineered_splits,
    feature_cols=resolved_features,
)

metrics_72h

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_72h,0.147531,0.457903,0.877835,14968,0.501203
1,val,y_72h,0.190859,0.561671,0.771524,1461,0.450376
2,test,y_72h,0.181356,0.540460,0.802660,2052,0.587232


In [13]:
all_metrics = pd.concat([metrics_24h, metrics_72h], ignore_index=True)
all_metrics

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.136491,0.429585,0.891705,14968,0.438268
1,val,y_24h,0.173140,0.519636,0.799518,1461,0.390144
2,test,y_24h,0.173973,0.523063,0.824129,2052,0.524366
3,train,y_72h,0.147531,0.457903,0.877835,14968,0.501203
4,val,y_72h,0.190859,0.561671,0.771524,1461,0.450376
5,test,y_72h,0.181356,0.540460,0.802660,2052,0.587232


In [14]:
baseline_metrics_path = PROJECT_ROOT / "reports" / "metrics" / "baselines_metrics.csv"
logreg_metrics_path = PROJECT_ROOT / "reports" / "metrics" / "logreg_baseline_metrics.csv"

baseline_metrics = pd.read_csv(baseline_metrics_path) if baseline_metrics_path.exists() else None
logreg_metrics = pd.read_csv(logreg_metrics_path) if logreg_metrics_path.exists() else None

print("XGB engineered metrics:")
display(all_metrics)

if baseline_metrics is not None:
    print("\nTeam baselines:")
    display(baseline_metrics)

if logreg_metrics is not None:
    print("\nYour logistic baseline:")
    display(logreg_metrics)

XGB engineered metrics:


,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.136491,0.429585,0.891705,14968,0.438268
1,val,y_24h,0.173140,0.519636,0.799518,1461,0.390144
2,test,y_24h,0.173973,0.523063,0.824129,2052,0.524366
3,train,y_72h,0.147531,0.457903,0.877835,14968,0.501203
4,val,y_72h,0.190859,0.561671,0.771524,1461,0.450376
5,test,y_72h,0.181356,0.540460,0.802660,2052,0.587232



Team baselines:


,model_name,split,horizon,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,climatology,train,24,0.246189,0.685506,0.500000,14968,0.438268
1,climatology,val,24,0.240248,0.673562,0.500000,1461,0.390144
2,climatology,test,24,0.256819,0.706875,0.500000,2052,0.524366
3,climatology,train,72,0.249999,0.693144,0.500000,14968,0.501203
4,climatology,val,72,0.250121,0.693389,0.500000,1461,0.450376
5,climatology,test,72,0.249792,0.692730,0.500000,2052,0.587232
6,simplified_rj,train,24,0.309720,0.843785,0.586920,14968,0.438268
7,simplified_rj,val,24,0.322588,0.870295,0.582776,1461,0.390144
8,simplified_rj,test,24,0.277449,0.766681,0.555566,2052,0.524366
9,simplified_rj,train,72,0.334306,0.936305,0.583603,14968,0.501203



Your logistic baseline:


,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.218343,0.632975,0.701915,14968,0.438268
1,val,y_24h,0.211554,0.611540,0.697125,1461,0.390144
2,test,y_24h,0.203154,0.594477,0.772720,2052,0.524366
3,train,y_72h,0.228040,0.650656,0.676093,14968,0.501203
4,val,y_72h,0.227992,0.647669,0.661882,1461,0.450376
5,test,y_72h,0.208006,0.602938,0.746592,2052,0.587232


In [15]:
print("Top 20 features for y_24h")
display(fi_24h.head(20))

print("\nTop 20 features for y_72h")
display(fi_72h.head(20))

Top 20 features for y_24h


,feature,importance,target
20,gcmt_depth_km,0.051136,y_24h
58,trigger_magnitude_sq,0.045461,y_24h
2,trigger_depth_km,0.043357,y_24h
31,gcmt_moment_exponent,0.041971,y_24h
59,prior_global_event_count_24h_log1p,0.039404,y_24h
57,trigger_depth_log1p,0.038836,y_24h
7,prior_global_event_count_24h,0.037085,y_24h
3,trigger_magnitude,0.032880,y_24h
25,gcmt_half_duration_sec,0.032598,y_24h
30,gcmt_depth_error_km,0.029763,y_24h



Top 20 features for y_72h


,feature,importance,target
2,trigger_depth_km,0.055209,y_72h
59,prior_global_event_count_24h_log1p,0.043006,y_72h
57,trigger_depth_log1p,0.042895,y_72h
3,trigger_magnitude,0.036138,y_72h
30,gcmt_depth_error_km,0.034535,y_72h
21,gcmt_magnitude,0.032914,y_72h
7,prior_global_event_count_24h,0.031365,y_72h
58,trigger_magnitude_sq,0.031327,y_72h
20,gcmt_depth_km,0.031182,y_72h
25,gcmt_half_duration_sec,0.030583,y_72h


In [16]:
output_dir = PROJECT_ROOT / "reports" / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

all_predictions = pd.concat([
    preds_24h["train"], preds_24h["val"], preds_24h["test"],
    preds_72h["train"], preds_72h["val"], preds_72h["test"],
], ignore_index=True)

all_feature_importance = pd.concat([fi_24h, fi_72h], ignore_index=True)

all_metrics.to_csv(output_dir / "xgb_engineered_metrics.csv", index=False)
all_predictions.to_csv(output_dir / "xgb_engineered_predictions.csv", index=False)
all_feature_importance.to_csv(output_dir / "xgb_engineered_feature_importance.csv", index=False)

print("Saved:")
print(output_dir / "xgb_engineered_metrics.csv")
print(output_dir / "xgb_engineered_predictions.csv")
print(output_dir / "xgb_engineered_feature_importance.csv")

Saved:
/Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/xgb_engineered_metrics.csv
/Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/xgb_engineered_predictions.csv
/Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/xgb_engineered_feature_importance.csv


# In comparison with our baselines

| Model           | ROC-AUC  | LogLoss  |
| --------------- | -------- | -------- |
| Climatology     | 0.50     | 0.71     |
| RJ baseline     | 0.56     | 0.77     |
| Logistic        | 0.77     | 0.59     |
| **XGB Model 1 y24** | **0.82** | **0.52** |


| Model           | ROC-AUC  | LogLoss  |
| --------------- | -------- | -------- |
| Climatology     | 0.50     | 0.69     |
| RJ baseline     | 0.55     | 0.81     |
| Logistic        | 0.75     | 0.60     |
| **XGB Model 1 y72** | **0.80** | **0.54** |


Our XGBoost model significantly outperforms both climatology and RJ-style baselines, achieving ROC-AUC scores of 0.82 (24h) and 0.80 (72h), indicating strong predictive performance and well-calibrated probabilities.